# Exploration of the HD model of sound responses of thalamic HD cells

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from scipy.optimize import minimize
from scipy.io import loadmat

In [ ]:
# Constants
N = 100 # 100 neurons
THETA = np.linspace(-np.pi, np.pi, N, endpoint = False) # Assignment of PD for the neurons

DT = 0.2 # temporal precision; default 0.5 ms
T = 500 # decay time for phasic response
T_BURN_IN = -50 # 50 ms for the model initialization to settle
T_END = 600
TIME = np.arange(T_BURN_IN, T_END, DT)

I_BASELINE = 34.27 # Baseline Activity drive
I_TUNING_CUE = 5 # external afferent input to visualize the Network tuning across all angles
SIGMA_CUE = np.deg2rad(15) # spatial tuning of the cue

FC_STIM_DURATION = 2 # 2ms of Step function Input
A_FAST = 800 # FC Activity drive
A_PHASIC = 60 # SC Activity drive
SIGMA_PHASIC = np.deg2rad(10) # spatial width for the phasic response

TAU_NEURAL = 10 # Neural Time constant
TAU_PHASIC_DECAY = 100 # Decay function for phasic response

T_STIM = 100 # Stimulus onset
T_PHASIC = 110 # Phasic onset at 55ms

J1 = 8.27 # Strength for local excitatory recurrent connections between cells with similar PD
J0 = 2.01 # Strength of uniform global inhibition
KAPPA = 3 # Higher = sharper tuning curve. Move from cosine connectivity to von Mises

W = np.zeros((N, N)) # Connectivity Matrix
tuning_heatmap = np.zeros((N, N))
times_to_plot = [90, 101, 110, 150]

#####
# checkpoint values
# I_BASELINE = 20
# J1 = 8.27
# J0 = 2.5
# KAPPA = 9
#####
#optimized
#--- Optimization complete (MSE = 0.94 Hz^2) ---
#  I_BASELINE       = 21.87
#  A_FAST           = 778.48
#  A_PHASIC         = 10.00
#  TAU_PHASIC_DECAY = 289.29 ms
#  J1               = 9.51
#  J0               = 3.87
#  t_delay          = 6.12 ms

In [ ]:
# Initialization of Connectivity matrix
for a in range(N):
    for b in range(N):
        d_theta = np.angle(
            np.exp(1j * (THETA[a] - THETA[b]))
        )
        W[a, b] = J1 * np.exp(KAPPA * (np.cos(d_theta) - 1.0)) - J0
W = W / N # Weight Normalziation

In [ ]:
plt.imshow(W, origin = 'lower', vmin = 0)
plt.xlabel('Neurons')
plt.ylabel('Neurons')
plt.colorbar(label = 'Connection Weight')
plt.show()

In [ ]:
# Connectivity visualization

def plot_mexican_hat(J1=4.0, J0=1.5):
    """
    Plots the Mexican Hat synaptic connectivity kernel.
    """
    # 1. Create the angular array (-180 to 180 degrees)
    theta_deg = np.linspace(-180, 180, 500)
    theta_rad = np.radians(theta_deg)
    
    # 2. Calculate the synaptic weight W
    # Formula: W(theta) = J1 * cos(theta) - J0
    W = J1 * np.cos(theta_rad) - J0
    
    # 3. Set up the plot
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Plot the main curve
    ax.plot(theta_deg, W, color='black', linewidth=2)
    
    # Add a zero-line to distinguish excitation from inhibition
    ax.axhline(0, color='gray', linestyle='--', linewidth=1.5)
    
    # Visually shade the excitatory (positive) and inhibitory (negative) regions
    ax.fill_between(theta_deg, W, 0, where=(W > 0), 
                    interpolate=True, color='tomato', alpha=0.4, label='Local Excitation (W > 0)')
    ax.fill_between(theta_deg, W, 0, where=(W <= 0), 
                    interpolate=True, color='royalblue', alpha=0.4, label='Global Inhibition (W < 0)')
    
    # 4. Formatting
    ax.set_title('Mexican Hat Synaptic Kernel\n$W(\\Delta\\theta) = J_1 \\cos(\\Delta\\theta) - J_0$', fontsize=14)
    ax.set_xlabel('Angular Distance $\\Delta\\theta$ (Degrees)', fontsize=12)
    ax.set_ylabel('Synaptic Weight ($W$)', fontsize=12)
    
    # Keep the X-axis fixed to a full circle
    ax.set_xlim(-180, 180)
    ax.set_xticks([-180, -90, 0, 90, 180])
    
    # Dynamically scale the Y-axis so the plot doesn't jump around too erratically
    max_height = max(10, J1)
    ax.set_ylim(-max_height - 2, max_height + 2)
    
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.show()

# 5. Connect the function to interactive sliders
#interact(plot_mexican_hat,
#         J1=FloatSlider(value=J1, min=0.0, max=10.0, step=0.1, description='J1 (Excitation):', style={'description_width': 'initial'}),
#         J0=FloatSlider(value=J0, min=0.0, max=15.0, step=0.1, description='J0 (Inhibition):', style={'description_width': 'initial'}));

In [ ]:
# investigation of tuning for created HD cells
for h_idx, heading in enumerate(THETA):
    # Flat initialization fo the network
    d_theta_cue = np.angle(np.exp(1j * (THETA - heading)))
    u = 40.0 * np.maximum(0, np.cos(d_theta_cue))
    r = np.maximum(0, u)

    I_ext = I_BASELINE + I_TUNING_CUE * np.exp(-0.5 * (d_theta_cue / SIGMA_CUE)**2)

    # Run integration until the bump stabilizes at the heading
    for t in TIME:
        u  += (-u + W @ r + I_ext) * (DT/TAU_NEURAL) # compute the neurons activity per time
        r = np.maximum(0, u)

    tuning_heatmap[:, h_idx] = r

fig = plt.figure(figsize=(14, 6))

# --- Panel 1: Cartesian Heatmap ---
# add_subplot(1, 2, 1) means: 1 row, 2 columns, 1st plot
ax1 = fig.add_subplot(1, 2, 2) 

im = ax1.imshow(
    tuning_heatmap,
    origin='lower',
    aspect='auto',
    extent=[-180, 180, -180, 180],
    cmap='magma',
    vmin=0, 
    vmax=np.max(tuning_heatmap),
)
ax1.set_title("Network Tuning Curve Heatmap\n(Steady-State Firing Rates)", fontsize=14)
ax1.set_xlabel("Current Heading / Stimulus Angle (°)", fontsize=12) # Fixed spelling
ax1.set_ylabel("Neuron Preferred Direction (PD) (°)", fontsize=12)  # Fixed from xlabel to ylabel
fig.colorbar(im, ax=ax1, label="Firing Rate (Hz)")

# --- Panel 2: Polar Plot ---
# add_subplot(1, 2, 2) means: 1 row, 2 columns, 2nd plot. 
# We explicitly set the projection to 'polar' here.
ax2 = fig.add_subplot(1, 2, 1, projection='polar')

# THETA is already in radians (-pi to pi), which Matplotlib's polar projection requires!
# Index 50 corresponds exactly to heading = 0 degrees
ax2.plot(THETA, tuning_heatmap[:, 50], color='tab:blue', linewidth=2)
ax2.set_title("Polar Tuning Curve\n(Heading = 0°)", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
def plot_model_full(I_baseline, A_fast, A_phasic, tau_decay, J1, J0, title_suffix=""):
    """Run CANN with given params and produce the full 4-panel diagnostic plot."""
    cos_d_theta = np.cos(np.angle(np.exp(1j * (THETA[:, None] - THETA[None, :]))))
    W_local = (J1 * np.exp(KAPPA * (cos_d_theta - 1.0)) - J0) / N

    u = 40 * np.maximum(0, np.cos(THETA))
    r = np.maximum(0, u)
    dRate_dt = np.zeros((len(TIME), N))

    for step, t in enumerate(TIME):
        I_ext = np.ones(N) * I_baseline
        if T_STIM <= t < (T_STIM + FC_STIM_DURATION):
            I_ext += A_fast
        elif t >= T_PHASIC:
            decay_factor = np.exp(-(t - T_PHASIC) / tau_decay)
            I_ext += A_phasic * decay_factor * np.exp(-0.5 * (THETA / SIGMA_PHASIC)**2)
        u += (-u + W_local @ r + I_ext) * (DT / TAU_NEURAL)
        r = np.maximum(0, u)
        dRate_dt[step, :] = r

    initialization_mask = TIME >= 0
    time_plot = TIME[initialization_mask]
    rates_plot = dRate_dt[initialization_mask, :]

    idx_0   = np.argmin(np.abs(THETA))
    idx_90  = np.argmin(np.abs(THETA - np.pi/2))
    idx_180 = np.argmin(np.abs(THETA - np.pi))

    fig = plt.figure(figsize=(12, 10))
    gs = fig.add_gridspec(2, 2)

    ax1 = fig.add_subplot(gs[0, :])
    im = ax1.imshow(
        rates_plot.T, aspect='auto',
        origin='lower',
        extent=[0, T_END, -180, 180],
        cmap='magma',
        vmin=0,
    )
    ax1.set_ylabel("Preferred Direction (deg)")
    ax1.set_title(f"Network Activity{title_suffix}")
    ax1.axvline(T_STIM, color='white', linestyle='--', alpha=0.5, label=f"Stim {T_STIM - 10} ms")
    ax1.axvline(T_PHASIC, color='cyan', linestyle='--', alpha=0.5, label=f"Rescue {T_PHASIC} ms")
    ax1.legend(loc='upper right')
    fig.colorbar(im, ax=ax1, label="Firing Rate (Hz)")

    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(time_plot, rates_plot[:, idx_0],   label="PD Cell (0°)",   color="red",    lw=2)
    ax2.plot(time_plot, rates_plot[:, idx_90],  label="PD Cell (90°)",  color="orange", lw=2)
    ax2.plot(time_plot, rates_plot[:, idx_180], label="PD Cell (180°)", color="blue",   lw=2)
    ax2.axvspan(0, T_STIM, color="gray", alpha=0.1, label="100ms Baseline")
    ax2.axvspan(T_STIM + FC_STIM_DURATION, T_PHASIC, color="red", alpha=0.1, label="Inhibitory Dip")
    ax2.set_xlabel("Time (ms)")
    ax2.set_ylabel("Firing Rate (Hz)")
    ax2.set_title(f"Temporal Traces (PSTH){title_suffix}")
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 1])
    snap_colors = ['black', 'red', 'blue', 'green']
    base_names = ['Baseline', 'FC', 'Inhibitory Dip', 'SC']
    snap_labels = [f"{name} ({time} ms)" for name, time in zip(base_names, times_to_plot)]
    for t_target, color, label in zip(times_to_plot, snap_colors, snap_labels):
        step_idx = int(t_target / DT)
        ax3.plot(np.rad2deg(THETA), rates_plot[step_idx, :], label=label, color=color, lw=2)
    ax3.set_xlabel("Preferred Direction (°)")
    ax3.set_ylabel("Firing Rate (Hz)")
    ax3.set_title(f"Spatial Tuning Snapshots{title_suffix}")
    ax3.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_model_full(I_BASELINE, A_FAST, A_PHASIC, TAU_PHASIC_DECAY, J1, J0)


In [ ]:
# load average firing curve

mat = loadmat('mean_7_rate.mat')

In [ ]:
bins_plot = mat['bins_plot'][0]
mean_seven_rate = mat['mean_seven_rate'][0]

## Temporal Trace Optimization to In Vivo Data

Fit model parameters so the PD cell at 0° matches the smoothed in vivo average response (`mean_seven_rate`).

**Free parameters:** `I_BASELINE`, `A_FAST`, `A_PHASIC`, `TAU_PHASIC_DECAY`, `J1`, `J0`, `t_delay`  
**Method:** `differential_evolution` (global, gradient-free)  
**Alignment:** model time relative to stim onset; in vivo bins shifted by `t_delay` (neural latency, ~20 ms) to align.

In [ ]:
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import differential_evolution
from scipy.interpolate import interp1d

# Smooth in vivo data with Gaussian kernel (sigma = 3 bins)
SMOOTH_SIGMA = 3
mean_seven_smooth = gaussian_filter1d(mean_seven_rate.astype(float), sigma=SMOOTH_SIGMA)

# Pre-compute fixed angular difference matrix so W rebuild inside optimizer is fast
D_THETA_MAT = np.angle(np.exp(1j * (THETA[:, None] - THETA[None, :])))
COS_D_THETA = np.cos(D_THETA_MAT)

IDX_0 = np.argmin(np.abs(THETA))  # PD cell at 0 degrees

# Comparison window (ms, relative to stim onset)
WIN_LO, WIN_HI = -50, 300

print(f'In vivo bins: {len(bins_plot)}, range {bins_plot[0]}..{bins_plot[-1]} ms')
print(f'FR range raw:      {mean_seven_rate.min():.1f}..{mean_seven_rate.max():.1f} Hz')
print(f'FR range smoothed: {mean_seven_smooth.min():.1f}..{mean_seven_smooth.max():.1f} Hz')


In [ ]:
def run_model_temporal(I_baseline, A_fast, A_phasic, tau_decay, J1, J0):
    """Run CANN; return (time_rel_to_stim, rates [time x N])."""
    W_local = (J1 * np.exp(KAPPA * (COS_D_THETA - 1.0)) - J0) / N

    u = 40.0 * np.maximum(0, np.cos(THETA))
    r = np.maximum(0, u)
    rates = np.zeros((len(TIME), N))

    for step, t in enumerate(TIME):
        I_ext = np.ones(N) * I_baseline
        if T_STIM <= t < (T_STIM + FC_STIM_DURATION):
            I_ext += A_fast
        elif t >= T_PHASIC:
            decay = np.exp(-(t - T_PHASIC) / tau_decay)
            I_ext += A_phasic * decay * np.exp(-0.5 * (THETA / SIGMA_PHASIC) ** 2)
        u += (-u + W_local @ r + I_ext) * (DT / TAU_NEURAL)
        r = np.maximum(0, u)
        rates[step, :] = r

    mask = TIME >= 0
    return TIME[mask] - T_STIM, rates[mask, :]  # time relative to stim onset


def loss_temporal(params):
    I_baseline, A_fast, A_phasic, tau_decay, J1, J0, t_delay = params

    t_model_rel, rates = run_model_temporal(
        I_baseline, A_fast, A_phasic, tau_decay, J1, J0
    )
    pd_trace = rates[:, IDX_0]

    # Shift in vivo by neural latency so stim onsets align
    t_vivo_rel = bins_plot.astype(float) - t_delay

    vivo_mask  = (t_vivo_rel  >= WIN_LO) & (t_vivo_rel  <= WIN_HI)
    model_mask = (t_model_rel >= WIN_LO) & (t_model_rel <= WIN_HI)

    if vivo_mask.sum() < 3 or model_mask.sum() < 3:
        return 1e10

    # Interpolate model onto the coarser in vivo time grid
    interp_fn = interp1d(
        t_model_rel[model_mask], pd_trace[model_mask],
        bounds_error=False, fill_value='extrapolate'
    )
    model_at_vivo = interp_fn(t_vivo_rel[vivo_mask])
    target        = mean_seven_smooth[vivo_mask]

    return float(np.mean((model_at_vivo - target) ** 2))


In [ ]:
# [I_baseline, A_fast, A_phasic, tau_decay, J1, J0, t_delay]
OPT_BOUNDS = [
    (5,    60),   # I_baseline  (Hz drive)
    (200, 2000),  # A_fast      (FC amplitude)
    (10,  300),   # A_phasic    (SC amplitude)
    (20,  500),   # tau_decay   (ms)
    (2,    25),   # J1          (local excitation strength)
    (0.1,  10),   # J0          (global inhibition strength)
    (0,    60),   # t_delay     (neural latency shift, ms)
]

print('Running differential_evolution (may take ~2-5 min)...')
opt_result = differential_evolution(
    loss_temporal,
    OPT_BOUNDS,
    seed=42,
    maxiter=200,
    popsize=15,
    tol=1e-5,
    mutation=(0.5, 1.5),
    recombination=0.7,
    disp=True,
    workers=1,
)

I_opt, Af_opt, Ap_opt, tau_opt, J1_opt, J0_opt, td_opt = opt_result.x
print(f'\n--- Optimization complete (MSE = {opt_result.fun:.2f} Hz^2) ---')
print(f'  I_BASELINE       = {I_opt:.2f}')
print(f'  A_FAST           = {Af_opt:.2f}')
print(f'  A_PHASIC         = {Ap_opt:.2f}')
print(f'  TAU_PHASIC_DECAY = {tau_opt:.2f} ms')
print(f'  J1               = {J1_opt:.2f}')
print(f'  J0               = {J0_opt:.2f}')
print(f'  t_delay          = {td_opt:.2f} ms')


differential_evolution step 128: f(x)= 1.5398822676509025


In [ ]:
t_model_rel, rates_opt = run_model_temporal(
    I_opt, Af_opt, Ap_opt, tau_opt, J1_opt, J0_opt
)
t_vivo_rel = bins_plot.astype(float) - td_opt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: temporal trace comparison
ax = axes[0]
ax.plot(t_vivo_rel, mean_seven_rate, color='gray', lw=1, alpha=0.4, label='In vivo (raw)')
ax.plot(t_vivo_rel, mean_seven_smooth, 'k-', lw=2, label='In vivo (smoothed)')
ax.plot(t_model_rel, rates_opt[:, IDX_0], 'r-', lw=2, label='Model PD (0 deg)')
ax.axvline(0, color='gray', linestyle='--', alpha=0.6, label='Stim onset')
ax.axvspan(WIN_LO, WIN_HI, color='lightblue', alpha=0.15, label='Fit window')
ax.set_xlim(-150, 450)
ax.set_xlabel('Time relative to stim (ms)')
ax.set_ylabel('Firing Rate (Hz)')
ax.set_title(f'Optimized Model vs In Vivo PD (0 deg)  [latency={td_opt:.1f} ms]')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: full network heatmap
ax2 = axes[1]
im = ax2.imshow(
    rates_opt.T, aspect='auto', origin='lower',
    extent=[t_model_rel[0], t_model_rel[-1], -180, 180],
    cmap='magma', vmin=0,
)
ax2.axvline(0, color='white', linestyle='--', alpha=0.5, label='Stim onset')
ax2.axvline(T_PHASIC - T_STIM, color='cyan', linestyle='--', alpha=0.5, label='SC onset')
ax2.set_xlabel('Time relative to stim (ms)')
ax2.set_ylabel('Preferred Direction (deg)')
ax2.set_title('Network Activity (Optimized)')
ax2.legend(loc='upper right')
fig.colorbar(im, ax=ax2, label='Firing Rate (Hz)')

plt.tight_layout()
plt.show()

print(f'Neural latency shift: {td_opt:.1f} ms')
print(f'Final MSE:            {opt_result.fun:.2f} Hz^2')


In [ ]:
plot_model_full(I_opt, Af_opt, Ap_opt, tau_opt, J1_opt, J0_opt, title_suffix=" (Optimized)")


## Parameter Space Explorer

Interactive J1/J0 exploration: 2D phase diagrams (peak rate, tuning width, bump existence) + live polar/linear tuning curve with sliders.

In [ ]:
# ─── Parameter Space Explorer ─────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

# Fallback to defaults if optimization hasn't run
_J1_ref  = globals().get('J1_opt',  J1)
_J0_ref  = globals().get('J0_opt',  J0)
_I_ref   = globals().get('I_opt',   I_BASELINE)

# Precompute angular difference matrix (reuse if already defined)
_COS_DT = globals().get('COS_D_THETA', np.cos(np.angle(np.exp(1j * (THETA[:, None] - THETA[None, :])))))

def _steady_state(J1v, J0v, I_base, n_steps=2000):
    W_ss = (J1v * np.exp(KAPPA * (_COS_DT - 1.0)) - J0v) / N
    u = 40.0 * np.maximum(0, np.cos(THETA))
    r = np.maximum(0, u)
    I_ext = np.ones(N) * I_base
    for _ in range(n_steps):
        u += (-u + W_ss @ r + I_ext) * (DT / TAU_NEURAL)
        r = np.maximum(0, u)
    return r

def _fwhm_deg(r):
    peak = r.max()
    if peak < 1e-6:
        return 0.0
    above = r >= peak / 2.0
    if above.sum() < 2:
        return 0.0
    idxs = np.where(above)[0]
    return np.rad2deg((idxs[-1] - idxs[0]) * (2 * np.pi / N))

# ── 1. Pre-compute J1/J0 grid ─────────────────────────────────────────────────
J1_SWEEP = np.linspace(1.0, 20.0, 35)
J0_SWEEP = np.linspace(0.1,  8.0, 35)

_peak  = np.zeros((len(J0_SWEEP), len(J1_SWEEP)))
_fwhm  = np.zeros((len(J0_SWEEP), len(J1_SWEEP)))
_bump  = np.zeros((len(J0_SWEEP), len(J1_SWEEP)))

print('Computing J1/J0 grid (~35×35 steady-state runs)...')
for j, J1v in enumerate(J1_SWEEP):
    for i, J0v in enumerate(J0_SWEEP):
        r_ss = _steady_state(J1v, J0v, _I_ref)
        _peak[i, j] = r_ss.max()
        _fwhm[i, j] = _fwhm_deg(r_ss)
        _bump[i, j] = float((r_ss.max() - r_ss.min()) > 2.0)
print('Done.')

# ── 2. J1/J0 phase diagrams ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ext = [J1_SWEEP[0], J1_SWEEP[-1], J0_SWEEP[0], J0_SWEEP[-1]]

def _mark_refs(ax):
    ax.scatter([_J1_ref], [_J0_ref], marker='*', s=250, color='yellow',
               edgecolors='black', zorder=6, label=f'Optimized ({_J1_ref:.1f}, {_J0_ref:.1f})')
    ax.scatter([J1], [J0], marker='o', s=120, color='white',
               edgecolors='black', zorder=6, label=f'Default ({J1}, {J0})')
    ax.legend(fontsize=8)
    ax.set_xlabel('J1 (local excitation)')
    ax.set_ylabel('J0 (global inhibition)')

im0 = axes[0].imshow(_peak, origin='lower', aspect='auto', extent=ext, cmap='hot')
axes[0].set_title('Peak Firing Rate (Hz)')
fig.colorbar(im0, ax=axes[0])
_mark_refs(axes[0])

im1 = axes[1].imshow(_fwhm, origin='lower', aspect='auto', extent=ext, cmap='viridis')
axes[1].set_title('Tuning FWHM (°)')
fig.colorbar(im1, ax=axes[1])
_mark_refs(axes[1])

im2 = axes[2].imshow(_bump, origin='lower', aspect='auto', extent=ext,
                     cmap='RdYlGn', vmin=0, vmax=1)
axes[2].set_title('Bump Exists (green=yes)')
fig.colorbar(im2, ax=axes[2], ticks=[0, 1])
_mark_refs(axes[2])

plt.suptitle('J1 / J0 Parameter Space  (I_baseline = {:.1f} Hz)'.format(_I_ref), fontsize=14)
plt.tight_layout()
plt.show()

# ── 3. Interactive tuning curve ───────────────────────────────────────────────
_theta_deg_fine = np.linspace(-180, 180, 500)
_theta_rad_fine = np.radians(_theta_deg_fine)

def _explore(J1v, J0v, I_base):
    r_ss  = _steady_state(J1v, J0v, I_base)
    fwhm  = _fwhm_deg(r_ss)
    kernel = J1v * np.exp(KAPPA * (np.cos(_theta_rad_fine) - 1.0)) - J0v

    fig = plt.figure(figsize=(15, 5))

    # --- kernel shape ---
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.plot(_theta_deg_fine, kernel, 'k', lw=2)
    ax1.axhline(0, color='gray', ls='--', lw=1)
    ax1.fill_between(_theta_deg_fine, kernel, 0, where=(kernel > 0),
                     color='tomato', alpha=0.4, label='Excitation')
    ax1.fill_between(_theta_deg_fine, kernel, 0, where=(kernel <= 0),
                     color='royalblue', alpha=0.4, label='Inhibition')
    ax1.set_title(f'Connectivity Kernel\nJ1={J1v:.1f}, J0={J0v:.1f}')
    ax1.set_xlabel('Δθ (°)')
    ax1.set_ylabel('W')
    ax1.set_xlim(-180, 180)
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # --- linear tuning curve ---
    ax2 = fig.add_subplot(1, 3, 2)
    ax2.plot(np.rad2deg(THETA), r_ss, color='steelblue', lw=2)
    if r_ss.max() > 1e-6:
        ax2.axhline(r_ss.max() / 2, color='orange', ls='--', alpha=0.8,
                    label=f'Half-max | FWHM = {fwhm:.0f}°')
    ax2.set_xlabel('Preferred Direction (°)')
    ax2.set_ylabel('Firing Rate (Hz)')
    ax2.set_title(f'Steady-State Tuning Curve\nPeak = {r_ss.max():.1f} Hz  |  FWHM = {fwhm:.0f}°')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    # --- polar tuning curve ---
    ax3 = fig.add_subplot(1, 3, 3, projection='polar')
    ax3.plot(THETA, r_ss, color='steelblue', lw=2)
    ax3.fill(THETA, r_ss, alpha=0.25, color='steelblue')
    ax3.set_title(f'Polar Tuning Curve\nJ1={J1v:.1f}, J0={J0v:.1f}, I={I_base:.0f}',
                  pad=20)

    plt.tight_layout()
    plt.show()

widgets.interact(
    _explore,
    J1v=widgets.FloatSlider(value=_J1_ref, min=1.0, max=20.0, step=0.1,
                            description='J1 (excit.):', style={'description_width': 'initial'}),
    J0v=widgets.FloatSlider(value=_J0_ref, min=0.1,  max=8.0,  step=0.1,
                            description='J0 (inhib.):', style={'description_width': 'initial'}),
    I_base=widgets.FloatSlider(value=_I_ref, min=5.0, max=60.0, step=0.5,
                               description='I_baseline:', style={'description_width': 'initial'}),
);